In [1]:
import json
from typing import Dict, Set, List
from functools import partial

import pandas as pd
import yaml
from IPython.display import display
from rapidfuzz import fuzz

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)

psg_directory = "../data/geography/"
psg_data_file = "psgc_2025-07-31.csv"

In [2]:
df = pd.read_csv(psg_directory + psg_data_file)
display(df.info())
display(df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43769 entries, 0 to 43768
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   psgc_id                43769 non-null  int64  
 1   name                   43769 non-null  object 
 2   correspondence_code    43719 non-null  float64
 3   geographic_level       43767 non-null  object 
 4   old_names              1699 non-null   object 
 5   city_class             149 non-null    object 
 6   income_classification  1724 non-null   object 
 7   settlement_type        42011 non-null  object 
 8   population             43762 non-null  object 
 9   Unnamed: 9             76 non-null     object 
 10  barangay_status        2855 non-null   object 
dtypes: float64(1), int64(1), object(9)
memory usage: 3.7+ MB


None

,psgc_id,name,correspondence_code,geographic_level,old_names,city_class,income_classification,settlement_type,population,Unnamed: 9,barangay_status
0,1300000000,National Capital Region (NCR),130000000.0,Reg,NaN,NaN,NaN,NaN,"13,484,462",NaN,NaN
1,1380100000,City of Caloocan,137501000.0,City,NaN,HUC,1st,NaN,"1,661,584",NaN,NaN
2,1380100001,Barangay 1,137501001.0,Bgy,NaN,NaN,NaN,U,"2,319",NaN,NaN
3,1380100002,Barangay 2,137501002.0,Bgy,NaN,NaN,NaN,U,"5,156",NaN,NaN
4,1380100003,Barangay 3,137501003.0,Bgy,NaN,NaN,NaN,U,"2,497",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
43764,1999908006,Manaulanan,124712037.0,Bgy,NaN,NaN,NaN,U,"7,632",NaN,NaN
43765,1999908007,Pamalian,124712062.0,Bgy,NaN,NaN,NaN,R,"3,256",NaN,NaN
43766,1999908008,Tapodoc,124717017.0,Bgy,NaN,NaN,NaN,R,"1,767",NaN,NaN
43767,1999908009,Macabual,124712034.0,Bgy,NaN,NaN,NaN,R,"4,557",NaN,NaN


In [3]:
# this code is just copied from my barangay project so there are more explanations there
# i think

df["psgc_id"] = df["psgc_id"].astype(str).str.zfill(10)
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

geographic_level_map = {
    "Reg": "region",
    "City": "city",
    "Mun": "municipality",
    "Prov": "province",
    "SubMun": "submunicipality",
    "Bgy": "barangay",
}
df["geographic_level"] = df["geographic_level"].replace(geographic_level_map)

df["barangay_code"] = df["psgc_id"].str[-3:]
df["municipal_or_city_code"] = df["psgc_id"].str[-5:-3]
df["province_or_huc_code"] = df["psgc_id"].str[-8:-5]
df["region_code"] = df["psgc_id"].str[-10:-8]

df["barangay_mapper"] = df["psgc_id"].str[-10:]
df["municipal_or_city_mapper"] = df["psgc_id"].str[-10:-3]
df["province_or_huc_mapper"] = df["psgc_id"].str[-10:-5]
df["region_mapper"] = df["psgc_id"].str[-10:-8]

df.sample(10)

regions_filter = (
    (df["province_or_huc_code"] == "000")
    & (df["municipal_or_city_code"] == "00")
    & (df["barangay_code"] == "000")
)
regions_mapper = (
    df.loc[regions_filter, ["region_mapper", "name"]]
    .sort_values("region_mapper")
    .set_index("region_mapper", drop=True)
    .to_dict()["name"]
)


province_or_huc_filter = (
    ~(df["province_or_huc_code"] == "000")
    & (df["municipal_or_city_code"] == "00")
    & (df["barangay_code"] == "000")
)

province_or_huc_mapper = (
    df.loc[province_or_huc_filter, ["province_or_huc_mapper", "name"]]
    .sort_values("province_or_huc_mapper")
    .set_index("province_or_huc_mapper")
    .to_dict()["name"]
)
municipal_or_city_filter = (
    ~(df["province_or_huc_code"] == "000")
    & ~(df["municipal_or_city_code"] == "00")
    & (df["barangay_code"] == "000")
)

municipal_or_city_mapper = (
    df.loc[municipal_or_city_filter, ["municipal_or_city_mapper", "name"]]
    .sort_values("municipal_or_city_mapper")
    .set_index("municipal_or_city_mapper")
    .to_dict()["name"]
)

df["region"] = df["region_mapper"].map(regions_mapper)
df["province_or_huc"] = df["province_or_huc_mapper"].map(province_or_huc_mapper)
df["municipality_or_city"] = df["municipal_or_city_mapper"].map(
    municipal_or_city_mapper
)
display(df.sample(10))

,psgc_id,name,correspondence_code,geographic_level,old_names,city_class,income_classification,settlement_type,population,Unnamed: 9,barangay_status,barangay_code,municipal_or_city_code,province_or_huc_code,region_code,barangay_mapper,municipal_or_city_mapper,province_or_huc_mapper,region_mapper,region,province_or_huc,municipality_or_city
4031,0102922009,Dammay,12922009.0,barangay,NaN,NaN,NaN,R,159,NaN,NaN,009,22,029,01,0102922009,0102922,01029,01,Region I (Ilocos Region),Ilocos Sur,Santa
7944,0203120016,Rangayan,23120016.0,barangay,NaN,NaN,NaN,R,777,NaN,NaN,016,20,031,02,0203120016,0203120,02031,02,Region II (Cagayan Valley),Isabela,Naguilian
21333,0600401000,Altavas,60401000.0,municipality,NaN,NaN,3rd,NaN,"25,639",NaN,NaN,000,01,004,06,0600401000,0600401,06004,06,Region VI (Western Visayas),Aklan,Altavas
22885,0603003020,Serallo,63003020.0,barangay,NaN,NaN,NaN,R,995,NaN,NaN,020,03,030,06,0603003020,0603003,06030,06,Region VI (Western Visayas),Iloilo,Anilao
23986,0603034015,Calam-isan,63034015.0,barangay,NaN,NaN,NaN,R,"1,318",NaN,NaN,015,34,030,06,0603034015,0603034,06030,06,Region VI (Western Visayas),Iloilo,Oton
19309,0501718046,Mantalisay,51718046.0,barangay,NaN,NaN,NaN,R,"1,610",NaN,NaN,046,18,017,05,0501718046,0501718,05017,05,Region V (Bicol Region),Camarines Sur,Libmanan
40136,1600203040,Tolosa,160203040.0,barangay,NaN,NaN,NaN,U,"5,998",NaN,NaN,040,03,002,16,1600203040,1600203,16002,16,Region XIII (Caraga),Agusan del Norte,City of Cabadbaran
12612,0401015021,San Miguel,41015021.0,barangay,NaN,NaN,NaN,R,769,NaN,NaN,021,15,010,04,0401015021,0401015,04010,04,Region IV-A (CALABARZON),Batangas,Lobo
10049,0304912003,Hilera,34912003.0,barangay,NaN,NaN,NaN,R,"1,961",NaN,NaN,003,12,049,03,0304912003,0304912,03049,03,Region III (Central Luzon),Nueva Ecija,Jaen
37063,1004215031,Owayan,104215031.0,barangay,NaN,NaN,NaN,R,374,NaN,NaN,031,15,042,10,1004215031,1004215,10042,10,Region X (Northern Mindanao),Misamis Occidental,City of Tangub


In [4]:
def sanitize_input(
    input_str: str | None, exclude: List[str] | str | None = None
) -> str:
    """
    Removes whitespaces, lowers, and remove all strings listed in exclude. If
    data is incompatible, will coerce to empty string.
    """
    if input_str is None:
        input_str = ""
    if not isinstance(input_str, str):
        input_str = ""
    sanitized_str = input_str.lower()
    if exclude is None:
        return sanitized_str

    if isinstance(exclude, list):
        exclude = [x.lower() for x in exclude if isinstance(x, str)]
        for item in exclude:
            sanitized_str = sanitized_str.replace(item, "")
        return sanitized_str

    return sanitized_str.replace(exclude.lower(), "")


cleanerjim = partial(
    sanitize_input, exclude=["(pob.)", "(pob)", ".", " ", "-", "(", ")", "&", "pob."]
)



In [5]:
bdf = df[df["geographic_level"] == "barangay"]

In [6]:
fuzzer_base = bdf[["name", "province_or_huc", "municipality_or_city", "psgc_id"]]

In [7]:
fuzzer_base = fuzzer_base.rename({"name": "barangay"}, axis=1)
fuzzer_base.to_parquet("../data/geography/fuzzer_base.parquet")

In [8]:
fuzzer_base

,barangay,province_or_huc,municipality_or_city,psgc_id
2,Barangay 1,City of Caloocan,NaN,1380100001
3,Barangay 2,City of Caloocan,NaN,1380100002
4,Barangay 3,City of Caloocan,NaN,1380100003
5,Barangay 4,City of Caloocan,NaN,1380100004
6,Barangay 5,City of Caloocan,NaN,1380100005
...,...,...,...,...
43764,Manaulanan,Special Geographic Area,Tugunan,1999908006
43765,Pamalian,Special Geographic Area,Tugunan,1999908007
43766,Tapodoc,Special Geographic Area,Tugunan,1999908008
43767,Macabual,Special Geographic Area,Tugunan,1999908009
